# 1: Import thư viện


In [ ]:

import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from collections import defaultdict

# Đường dẫn file - sửa lại nếu cần
TRAIN_PATH = "train_logs_program.parquet"
TEST_EPG_PATH = "test_epg_candidates.parquet"
SUBMISSION_PATH = "submission.csv"


# 2: Đọc dữ liệu


In [ ]:

train_df = pd.read_parquet(TRAIN_PATH)
test_epg = pd.read_parquet(TEST_EPG_PATH)
sub_df = pd.read_csv(SUBMISSION_PATH)

print("Train shape:", train_df.shape)
display(train_df.head())

print("Test EPG shape:", test_epg.shape)
display(test_epg.head())

print("Submission shape:", sub_df.shape)
display(sub_df.head())

Train shape: (1202549, 6)


,user_id,tv_show_id,vsetv_id,start_time_view,end_time_view,duration_view
0,57357915868236403,6600437,353,2020-03-09 07:39:35,2020-03-09 07:44:41,306
1,8404698046253197367,6600437,353,2020-03-09 07:40:45,2020-03-09 08:06:41,1556
2,10561746661310954575,6600437,353,2020-03-09 07:40:48,2020-03-09 07:46:54,366
3,5102444605050899938,6600437,353,2020-03-09 07:48:30,2020-03-09 08:05:55,1045
4,1725562967272176802,6600437,353,2020-03-09 07:54:15,2020-03-09 07:58:09,234


Test EPG shape: (144774, 4)


,channel_id,tv_show_id,start_ts,end_ts
0,3,20088,1595831400,1595833200
1,3,2400480,1595833200,1595833800
2,3,20088,1595833800,1595836800
3,3,2400480,1595836800,1595837400
4,3,20088,1595837400,1595840400


Submission shape: (1260, 2)


,user_id,tv_show_id
0,8377619604347126107,0 0 0 0 0
1,8381667675275833309,0 0 0 0 0
2,8387147770138767246,0 0 0 0 0
3,8397181578236218580,0 0 0 0 0
4,8404698046253197367,0 0 0 0 0


# 3: Tạo dữ liệu user-item từ train_logs_program


In [ ]:

# Nếu còn tv_show_id = 0 thì bỏ (phòng trường hợp)
train_df = train_df[train_df["tv_show_id"] != 0]

# Gộp theo (user_id, tv_show_id)
agg = (
    train_df
    .groupby(["user_id", "tv_show_id"], as_index=False)["duration_view"]
    .sum()
    .rename(columns={"duration_view": "watch_time"})
)

# Rating implicit: log(1 + thời lượng xem)
agg["rating"] = np.log1p(agg["watch_time"])

print("Số dòng tương tác (user, tv_show):", len(agg))
agg.head()


Số dòng tương tác (user, tv_show): 627456


,user_id,tv_show_id,watch_time,rating
0,2244466330591177,200405,8,2.197225
1,2244466330591177,240081,2344,7.760041
2,2244466330591177,400335,5942,8.689969
3,2244466330591177,700369,643,6.467699
4,2244466330591177,1000384,3401,8.132119


# 4: Mã hoá user_id, tv_show_id  & tạo ma trận CSR


In [ ]:

# Danh sách id
user_ids = agg["user_id"].unique()
item_ids = agg["tv_show_id"].unique()

print("Số user:", len(user_ids))
print("Số item (tv_show):", len(item_ids))

# Mapping id -> index
user2idx = {uid: idx for idx, uid in enumerate(user_ids)}
item2idx = {iid: idx for idx, iid in enumerate(item_ids)}

idx2user = {idx: uid for uid, idx in user2idx.items()}
idx2item = {idx: iid for iid, idx in item2idx.items()}

# Ánh xạ sang index
agg["user_idx"] = agg["user_id"].map(user2idx)
agg["item_idx"] = agg["tv_show_id"].map(item2idx)

n_users = len(user_ids)
n_items = len(item_ids)

# Tạo ma trận user-item (CSR)
user_item_csr = csr_matrix(
    (agg["rating"].values, (agg["user_idx"].values, agg["item_idx"].values)),
    shape=(n_users, n_items)
)

print("user_item_csr shape:", user_item_csr.shape)


Số user: 4885
Số item (tv_show): 4160
user_item_csr shape: (4885, 4160)


# 5: Candidate tv_show_id từ test + độ phổ biến toàn cục


In [ ]:

# Bỏ tv_show_id = 0 nếu có
candidate_tv_ids = test_epg.loc[test_epg["tv_show_id"] != 0, "tv_show_id"].unique().tolist()
candidate_tv_set = set(candidate_tv_ids)

print("Số tv_show candidate (test):", len(candidate_tv_ids))

# Tính độ phổ biến item theo tổng rating (xem nhiều -> phổ biến)
item_pop = (
    agg.groupby("tv_show_id")["rating"]
    .sum()
    .sort_values(ascending=False)
)

popular_item_ids = item_pop.index.tolist()

# Độ phổ biến *trên tập candidate*
popular_candidate_ids = [iid for iid in popular_item_ids if iid in candidate_tv_set]

# Nếu ít hơn 5 thì bổ sung thêm từ candidate_tv_ids cho đủ
if len(popular_candidate_ids) < 5:
    extra = [iid for iid in candidate_tv_ids if iid not in popular_candidate_ids]
    popular_candidate_ids.extend(extra)

popular_candidate_ids = popular_candidate_ids[:50]  # lấy top 50 để dùng fallback

print("Top 10 candidate phổ biến:", popular_candidate_ids[:10])


Số tv_show candidate (test): 6636
Top 10 candidate phổ biến: [2400480, 6500479, 240081, 200352, 2400508, 400335, 500315, 2500430, 400361, 20088]


# 6: Train Item-based CF với NearestNeighbors (cosine KNN)


In [ ]:

# Ma trận item-user (mỗi item là 1 vector user)
item_user_csr = user_item_csr.T  # shape = (n_items, n_users)

print("item_user_csr shape:", item_user_csr.shape)

# Số hàng xóm mỗi item (K)
K = 50
n_neighbors = min(K + 1, n_items)  # +1 vì sẽ bao gồm chính nó

nn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=n_neighbors,
    n_jobs=-1
)

nn_model.fit(item_user_csr)

# distances, indices cho TẤT CẢ item
distances, indices = nn_model.kneighbors(item_user_csr, return_distance=True)

# Chuyển về similarity: sim = 1 - distance
sims = 1.0 - distances
sims[sims < 0] = 0.0  # cắt âm (do sai số số học)

# Bỏ chính nó ở vị trí đầu (k=0)
item_neighbors = indices[:, 1:]        # shape: (n_items, K)
item_neighbors_sims = sims[:, 1:]      # shape: (n_items, K)

print("item_neighbors shape:", item_neighbors.shape)
print("item_neighbors_sims shape:", item_neighbors_sims.shape)


item_user_csr shape: (4160, 4885)
item_neighbors shape: (4160, 50)
item_neighbors_sims shape: (4160, 50)


# 7: Hàm gợi ý top-N tv_show_id cho 1 user_id


In [ ]:

candidate_tv_set = set(candidate_tv_ids)  # để kiểm tra nhanh


def recommend_for_user_itemcf(user_id, N=5):
    """
    Trả về list N tv_show_id gợi ý cho user_id.
    - Nếu user chưa từng xuất hiện trong train -> dùng global popularity trên candidate.
    - Nếu không có score nào từ CF -> fallback popularity.
    """
    # User chưa có trong train
    if user_id not in user2idx:
        return popular_candidate_ids[:N]

    u_idx = user2idx[user_id]
    # Hàng tương ứng user u
    user_row = user_item_csr[u_idx]  # CSR (1, n_items)

    # Các item user đã xem
    items_u_idx = user_row.indices
    ratings_u = user_row.data

    if len(items_u_idx) == 0:
        return popular_candidate_ids[:N]

    items_u_idx_set = set(items_u_idx)

    scores = defaultdict(float)

    # Duyệt từng item i mà user đã xem
    for i_idx, r_ui in zip(items_u_idx, ratings_u):
        # Hàng xóm của item i
        neigh_indices = item_neighbors[i_idx]
        neigh_sims = item_neighbors_sims[i_idx]

        for j_idx, sim_ij in zip(neigh_indices, neigh_sims):
            if sim_ij <= 0:
                continue

            # Không recommend item đã xem
            if j_idx in items_u_idx_set:
                continue

            tv_j = idx2item[j_idx]

            # Chỉ xét item thuộc tập candidate test
            if tv_j not in candidate_tv_set:
                continue

            # Cộng dồn score theo Item-based CF: score(u, j) += sim(i, j) * r_ui
            scores[j_idx] += sim_ij * r_ui

    # Nếu không có score -> fallback
    if not scores:
        return popular_candidate_ids[:N]

    # Sắp xếp theo score giảm dần
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    rec_tv_ids = []
    for j_idx, s in sorted_scores:
        tv_j = idx2item[j_idx]
        if tv_j in rec_tv_ids:
            continue
        rec_tv_ids.append(tv_j)
        if len(rec_tv_ids) >= N:
            break

    # Nếu vẫn thiếu thì pad thêm từ popularity
    if len(rec_tv_ids) < N:
        for tv_j in popular_candidate_ids:
            if tv_j not in rec_tv_ids:
                rec_tv_ids.append(tv_j)
            if len(rec_tv_ids) >= N:
                break

    return rec_tv_ids[:N]


# 8: Sinh cột tv_show_id (chuỗi "id1 id2 id3 id4 id5") cho submission


In [ ]:

def format_recs_as_string(tv_ids):
    # Chuyển list [123, 456, ...] -> "123 456 ..."
    return " ".join(str(int(t)) for t in tv_ids)


pred_list = []

for uid in sub_df["user_id"]:
    recs = recommend_for_user_itemcf(uid, N=5)
    pred_list.append(format_recs_as_string(recs))

sub_df["tv_show_id"] = pred_list

# Lưu file submission mới
OUTPUT_PATH = "submission_item_based_cf.csv"
sub_df.to_csv(OUTPUT_PATH, index=False)

print("Đã lưu file:", OUTPUT_PATH)
sub_df

Đã lưu file: submission_item_based_cf.csv


,user_id,tv_show_id
0,8377619604347126107,200405 1000723 300358 2500261 400363
1,8381667675275833309,2400508 2500430 20088 1100411 5800509
2,8387147770138767246,400335 500315 240081 20088 2500430
3,8397181578236218580,2400508 400335 200469 200337 500346
4,8404698046253197367,500331 1000723 6000483 2500413 400426
